In [1]:
!pip install mlflow xgboost --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 726.7 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 40.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 10.2 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from xgboost import XGBClassifier
from datetime import datetime
import time

print("✅ All imports done!")

✅ All imports done!


In [6]:
# ── Dataset (same seed as your Week 13) ───────────────────────────
np.random.seed(42)
n_samples = 10000

temperature = np.random.normal(75, 15, n_samples)
vibration   = np.random.normal(0.5, 0.2, n_samples)
pressure    = np.random.normal(100, 20, n_samples)
rpm         = np.random.normal(1500, 200, n_samples)
age_days    = np.random.randint(0, 365, n_samples)

failure_score = (
    (temperature > 90) * 0.3 +
    (vibration > 0.8)  * 0.3 +
    (pressure > 130)   * 0.2 +
    (age_days > 300)   * 0.2
)
failure_prob = failure_score + np.random.normal(0, 0.1, n_samples)
failure = (failure_prob > 0.5).astype(int)

data = pd.DataFrame({
    'temperature': temperature,
    'vibration':   vibration,
    'pressure':    pressure,
    'rpm':         rpm,
    'age_days':    age_days,
    'failure':     failure
})

print(f"Dataset shape: {data.shape}")
print(f"Failure rate:  {data.failure.mean():.2%}")

# ── Train/Test Split ───────────────────────────────────────────────
X = data.drop('failure', axis=1)
y = data['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")

# ── MLflow Setup (SQLite — fixes MLflow 3.x error) ────────────────
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("predictive-maintenance")

print(f"\nTracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment:   predictive-maintenance")

# ── Train & Log All 3 Models ───────────────────────────────────────
print("\nTraining all 3 models...\n")

# Logistic Regression
with mlflow.start_run(run_name="logistic_regression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("max_iter", 1000)
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, y_train)
    y_pred  = lr.predict(X_test_scaled)
    y_proba = lr.predict_proba(X_test_scaled)[:, 1]
    lr_metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1_score":  f1_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba)
    }
    mlflow.log_metrics(lr_metrics)
    mlflow.sklearn.log_model(lr, "model")
    print(f"Logistic Regression → ROC AUC: {lr_metrics['roc_auc']:.4f}")

# Random Forest
with mlflow.start_run(run_name="random_forest"):
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("min_samples_split", 5)
    rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                min_samples_split=5, random_state=42)
    rf.fit(X_train_scaled, y_train)
    y_pred  = rf.predict(X_test_scaled)
    y_proba = rf.predict_proba(X_test_scaled)[:, 1]
    rf_metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1_score":  f1_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba)
    }
    mlflow.log_metrics(rf_metrics)
    mlflow.sklearn.log_model(rf, "model")
    print(f"Random Forest      → ROC AUC: {rf_metrics['roc_auc']:.4f}")

# XGBoost
with mlflow.start_run(run_name="xgboost"):
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                        random_state=42, eval_metric="logloss")
    xgb.fit(X_train_scaled, y_train)
    y_pred  = xgb.predict(X_test_scaled)
    y_proba = xgb.predict_proba(X_test_scaled)[:, 1]
    xgb_metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1_score":  f1_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba)
    }
    mlflow.log_metrics(xgb_metrics)
    mlflow.sklearn.log_model(xgb, "model")
    print(f"XGBoost            → ROC AUC: {xgb_metrics['roc_auc']:.4f}")

print("\n✅ All 3 models trained and logged!")

Dataset shape: (10000, 6)
Failure rate:  4.15%

Train: (8000, 5) | Test: (2000, 5)


2026/06/07 10:04:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 10:04:29 INFO mlflow.store.db.utils: Updating database tables
2026/06/07 10:04:31 INFO mlflow.tracking.fluent: Experiment with name 'predictive-maintenance' does not exist. Creating a new experiment.



Tracking URI: sqlite:////kaggle/working/mlflow.db
Experiment:   predictive-maintenance

Training all 3 models...



2026/06/07 10:04:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 10:04:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression → ROC AUC: 0.9234


2026/06/07 10:04:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 10:04:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest      → ROC AUC: 0.9751


2026/06/07 10:04:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 10:04:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost            → ROC AUC: 0.9709

✅ All 3 models trained and logged!


In [7]:
# Setup client
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
client = MlflowClient()

experiment = client.get_experiment_by_name("predictive-maintenance")
print(f"Experiment ID: {experiment.experiment_id}\n")

# Fetch all runs sorted by ROC AUC
runs = client.search_runs(
    experiment_ids=experiment.experiment_id,
    order_by=["metrics.roc_auc DESC"]
)

# Display clean table
run_data = []
for run in runs:
    run_data.append({
        "run_id":     run.info.run_id[:8],
        "name":       run.info.run_name,
        "model_type": run.data.params.get("model_type", "Unknown"),
        "roc_auc":    round(run.data.metrics.get("roc_auc", 0), 4),
        "f1_score":   round(run.data.metrics.get("f1_score", 0), 4),
        "accuracy":   round(run.data.metrics.get("accuracy", 0), 4)
    })

df = pd.DataFrame(run_data)
print("All Runs (sorted by ROC AUC):")
print(df.to_string(index=False))
print(f"\n🏆 Best model: {df.iloc[0]['name']} → ROC AUC {df.iloc[0]['roc_auc']}")

Experiment ID: 1

All Runs (sorted by ROC AUC):
  run_id                name         model_type  roc_auc  f1_score  accuracy
c8d59c97       random_forest       RandomForest   0.9751    0.5976     0.967
cf6899c2             xgboost            XGBoost   0.9709    0.5926     0.967
1bbc70c1 logistic_regression LogisticRegression   0.9234    0.2549     0.962

🏆 Best model: random_forest → ROC AUC 0.9751


In [8]:
best_run    = runs[0]
best_run_id = best_run.info.run_id

print(f"Best model:  {best_run.info.run_name}")
print(f"ROC AUC:     {best_run.data.metrics['roc_auc']:.4f}")
print(f"Run ID:      {best_run_id[:8]}\n")

model_name = "PredictiveMaintenance"
model_uri  = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

time.sleep(2)
print(f"✅ Registered '{model_name}' — Version {registered_model.version}")

Successfully registered model 'PredictiveMaintenance'.
2026/06/07 10:06:04 WARNING mlflow.tracking._model_registry.fluent: Run with id c8d59c9762f24b8ab7f85e06d1b90150 has no artifacts at artifact path 'model', registering model based on models:/m-a794a618d1f44d41b8d5e0a5fe3e07ef instead


Best model:  random_forest
ROC AUC:     0.9751
Run ID:      c8d59c97



Created version '1' of model 'PredictiveMaintenance'.


✅ Registered 'PredictiveMaintenance' — Version 1


In [9]:
version    = registered_model.version
roc_auc    = best_run.data.metrics["roc_auc"]
f1         = best_run.data.metrics["f1_score"]
accuracy   = best_run.data.metrics["accuracy"]
model_type = best_run.data.params.get("model_type", "Unknown")

description = f"""
{model_type} model for predictive maintenance.
Trained on equipment sensor data (10,000 samples).

Performance:
- ROC AUC:  {roc_auc:.4f}
- F1 Score: {f1:.4f}
- Accuracy: {accuracy:.4f}

Features: temperature, vibration, pressure, rpm, age_days
Training split: 80/20 | Random seed: 42
"""

client.update_model_version(
    name=model_name,
    version=version,
    description=description
)

client.set_model_version_tag(model_name, version, "validation_status", "passed")
client.set_model_version_tag(model_name, version, "team",              "datascience")
client.set_model_version_tag(model_name, version, "framework",         model_type.lower())

print("✅ Documentation and tags added!")
print(description)

✅ Documentation and tags added!

RandomForest model for predictive maintenance.
Trained on equipment sensor data (10,000 samples).

Performance:
- ROC AUC:  0.9751
- F1 Score: 0.5976
- Accuracy: 0.9670

Features: temperature, vibration, pressure, rpm, age_days
Training split: 80/20 | Random seed: 42



In [10]:
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage="Staging",
    archive_existing_versions=True
)
print(f"✅ Version {version} moved to Staging\n")

# Load staging model
staging_model = mlflow.pyfunc.load_model(
    model_uri=f"models:/{model_name}/Staging"
)
print("Staging model loaded successfully!\n")

# Test high-risk scenario
test_data = pd.DataFrame({
    "temperature": [95],
    "vibration":   [0.9],
    "pressure":    [135],
    "rpm":         [1500],
    "age_days":    [320]
})

prediction = staging_model.predict(test_data)
print("Staging Test — High Risk Equipment:")
print(f"  temp=95, vibration=0.9, pressure=135, age=320 days")
print(f"  Prediction: {'⚠️  FAILURE LIKELY' if prediction[0] == 1 else '✅ NO FAILURE'}")

✅ Version 1 moved to Staging

Staging model loaded successfully!

Staging Test — High Risk Equipment:
  temp=95, vibration=0.9, pressure=135, age=320 days
  Prediction: ⚠️  FAILURE LIKELY


/tmp/ipykernel_58/3122978058.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [11]:
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage="Production",
    archive_existing_versions=True
)

client.set_model_version_tag(
    model_name, version,
    "deployment_date",
    datetime.now().strftime("%Y-%m-%d")
)

print(f"🚀 Version {version} is now in PRODUCTION!")
print(f"   Model:    {model_type}")
print(f"   ROC AUC:  {roc_auc:.4f}")
print(f"   Deployed: {datetime.now().strftime('%Y-%m-%d')}")

🚀 Version 1 is now in PRODUCTION!
   Model:    RandomForest
   ROC AUC:  0.9751
   Deployed: 2026-06-07


/tmp/ipykernel_58/2704849971.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [12]:
def predict_equipment_failure(temperature, vibration, pressure, rpm, age_days):
    """
    Production inference function.
    Always loads the latest Production model automatically.
    """
    model = mlflow.pyfunc.load_model(
        model_uri=f"models:/{model_name}/Production"
    )
    input_data = pd.DataFrame([{
        "temperature": temperature,
        "vibration":   vibration,
        "pressure":    pressure,
        "rpm":         rpm,
        "age_days":    age_days
    }])
    prediction = model.predict(input_data)[0]
    return {
        "will_fail":      bool(prediction),
        "recommendation": "⚠️  Schedule maintenance" if prediction else "✅ Normal operation"
    }

# Test 3 scenarios
scenarios = [
    {"name": "Normal",    "temp": 70, "vib": 0.4, "press": 95,  "rpm": 1500, "age": 100},
    {"name": "High Risk", "temp": 95, "vib": 0.9, "press": 135, "rpm": 1500, "age": 320},
    {"name": "Medium",    "temp": 85, "vib": 0.6, "press": 110, "rpm": 1500, "age": 200}
]

print("Production Predictions:\n")
print(f"{'Scenario':<12} {'Temp':>5} {'Vib':>5} {'Press':>6} {'Age':>5}  Result")
print("-" * 60)
for s in scenarios:
    result = predict_equipment_failure(
        s["temp"], s["vib"], s["press"], s["rpm"], s["age"]
    )
    print(f"{s['name']:<12} {s['temp']:>5} {s['vib']:>5} {s['press']:>6} {s['age']:>5}  {result['recommendation']}")

Production Predictions:

Scenario      Temp   Vib  Press   Age  Result
------------------------------------------------------------
Normal          70   0.4     95   100  ⚠️  Schedule maintenance
High Risk       95   0.9    135   320  ⚠️  Schedule maintenance
Medium          85   0.6    110   200  ⚠️  Schedule maintenance


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [13]:
# Register second-best model as version 2
second_run   = runs[1]
model_uri_v2 = f"runs:/{second_run.info.run_id}/model"

v2 = mlflow.register_model(model_uri_v2, model_name)
time.sleep(2)

print(f"✅ Registered Version {v2.version}")
print(f"   Model:   {second_run.data.params.get('model_type', 'Unknown')}")
print(f"   ROC AUC: {second_run.data.metrics.get('roc_auc', 0):.4f}\n")

# Rollback to version 1
print("Simulating rollback to version 1...")
client.transition_model_version_stage(
    name=model_name,
    version="1",
    stage="Production",
    archive_existing_versions=True
)
print("✅ Rollback complete! Production is back to version 1.\n")

# Confirm current production version
prod = client.get_latest_versions(model_name, stages=["Production"])
prod_run = client.get_run(prod[0].run_id)
print(f"Current Production version: {prod[0].version}")
print(f"Model type: {prod_run.data.params.get('model_type', 'Unknown')}")
print(f"ROC AUC:    {prod_run.data.metrics.get('roc_auc', 0):.4f}")

Registered model 'PredictiveMaintenance' already exists. Creating a new version of this model...
2026/06/07 10:07:54 WARNING mlflow.tracking._model_registry.fluent: Run with id cf6899c2758a40ff9d1729eba6feba00 has no artifacts at artifact path 'model', registering model based on models:/m-018f743fe80a473199507e75678495a1 instead
Created version '2' of model 'PredictiveMaintenance'.


✅ Registered Version 2
   Model:   XGBoost
   ROC AUC: 0.9709

Simulating rollback to version 1...
✅ Rollback complete! Production is back to version 1.

Current Production version: 1
Model type: RandomForest
ROC AUC:    0.9751


/tmp/ipykernel_58/2706538882.py:14: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
/tmp/ipykernel_58/2706538882.py:23: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  prod = client.get_latest_versions(model_name, stages=["Production"])


In [14]:
print("=" * 55)
print("   WEEK 14 — MODEL REGISTRY COMPLETE")
print("=" * 55)

all_versions = client.search_model_versions(f"name='{model_name}'")
for v in sorted(all_versions, key=lambda x: int(x.version)):
    run = client.get_run(v.run_id)
    print(f"\n  Version {v.version}  [{v.current_stage}]")
    print(f"    Model:   {run.data.params.get('model_type', 'Unknown')}")
    print(f"    ROC AUC: {run.data.metrics.get('roc_auc', 0):.4f}")

print("\n\nDeliverables completed:")
print("  ✅ MlflowClient configured")
print("  ✅ All runs displayed with metrics")
print("  ✅ Best model registered")
print("  ✅ Description and tags added")
print("  ✅ Transitioned to Staging")
print("  ✅ Staging model tested")
print("  ✅ Promoted to Production")
print("  ✅ predict_equipment_failure() built and tested")
print("  ✅ Version 2 registered")
print("  ✅ Rollback tested successfully")

   WEEK 14 — MODEL REGISTRY COMPLETE

  Version 1  [Production]
    Model:   RandomForest
    ROC AUC: 0.9751

  Version 2  [None]
    Model:   XGBoost
    ROC AUC: 0.9709


Deliverables completed:
  ✅ MlflowClient configured
  ✅ All runs displayed with metrics
  ✅ Best model registered
  ✅ Description and tags added
  ✅ Transitioned to Staging
  ✅ Staging model tested
  ✅ Promoted to Production
  ✅ predict_equipment_failure() built and tested
  ✅ Version 2 registered
  ✅ Rollback tested successfully


In [15]:
import shutil

# Copy the MLflow database to output folder
shutil.copy("/kaggle/working/mlflow.db", "/kaggle/working/mlflow_week14.db")
print("✅ mlflow_week14.db saved — download from Output panel")

✅ mlflow_week14.db saved — download from Output panel
